In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from langchain_gigachat import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [2]:
from src.main.prompts.text import MERGE_PROMPT, FEATURE_PROMPT
from src.main.utils.traceback_extractor import extract_exec_error
from src.main.utils.atrifact_saver import ArtifactSaver

In [3]:
llm = GigaChat(
        model="GigaChat-2-Max",
        verify_ssl_certs=False, 
        profanity_check=False,
        credentials=os.getenv("GIGACHAT_CREDENTIALS"),
        scope="GIGACHAT_API_PERS",
        temperature=0.3,
        max_tokens=4096
    )

In [4]:
def extract_code(text: str) -> str:
    match = re.search(r"```(?:python)?\n(.*?)\n```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

In [ ]:
def merge_phase(task_desc: str, data_dir: str, train_df: pd.DataFrame, test_df: pd.DataFrame, llm: GigaChat):
    # 1. Собираем схему без лишних деталей
    schema = []
    for f in sorted(Path(data_dir).glob("*.csv")):
        if f.name in ("train.csv", "test.csv"): continue
        df = pd.read_csv(f, nrows=3, low_memory=False)
        schema.append(f"📄 {f.name}\nКолонки: {list(df.columns)}\nПример:\n{df.head(2).to_string()}\n")
    
    # 2. Инициализируем диалог
    messages = [
        SystemMessage(content=MERGE_PROMPT),
        HumanMessage(content=f"Задача: {task_desc}\n\nСхема данных:\n{''.join(schema)}")
    ]

    # 3. Цикл генерации → исполнение → фидбек
    for attempt in range(1, 6):
        print(f"\nПопытка {attempt}/5")
        response = llm.invoke(messages)
        code = extract_code(str(response.content) if hasattr(response, 'content') else str(response))

        print(code)
        
        # Сохраняем ответ LLM в историю
        messages.append(AIMessage(content=code))
        
        try:
            # Чистое пространство имён, только pandas/numpy
            ns = {"pd": pd, "np": np, "Path": Path}
            exec(code, ns)
            func = ns["merge_data"]
            
            merged_train, merged_test = func(train_df.copy(), test_df.copy(), data_dir)
            print(f"Успех! Train: {merged_train.shape}, Test: {merged_test.shape}")
            return merged_train, merged_test, code  # Возвращаем код для повторного использования
            
        except Exception as e:
            line_no, error_line = extract_exec_error(code, e)
            err_text = f"❌ Ошибка '{type(e).__name__}: {e}' в строке {line_no}:" + repr(error_line)
            print(err_text)
            # Кидаем traceback обратно в контекст, LLM исправит
            messages.append(HumanMessage(content=f"{err_text}\nИсправь код и верни заново. Ни в коем случае не допускай ту же ошибку еще раз. Будь внимательнее к задаче. ОБЯЗАТЕЛЬНО вначале напиши комментарий почему ты допустил ошибку и как будешь ее исправлять."))
        
            
    raise RuntimeError("Не удалось получить рабочий merge-код за 5 попыток")

In [6]:
with open("data/readme.txt", "r", encoding="utf-8") as file:
    description = "\n".join(file.readlines())

In [7]:
py_code_saver = ArtifactSaver("artifacts", "py")

In [8]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

m_train, m_test, merge_code = merge_phase(
    task_desc=description,
    data_dir="data",
    train_df=train,
    test_df=test,
    llm=llm
)


🔄 Попытка 1/5
def merge_data(train_df, test_df, data_dir):
    # Load additional datasets
    users = pd.read_csv(f'{data_dir}/users.csv')
    orders = pd.read_csv(f'{data_dir}/orders.csv').query("eval_set == 'prior'")
    order_items = pd.read_csv(f'{data_dir}/order_items.csv')
    products = pd.read_csv(f'{data_dir}/products.csv')
    aisles = pd.read_csv(f'{data_dir}/aisles.csv')
    departments = pd.read_csv(f'{data_dir}/departments.csv')
    
    # Aggregate order items by user and product to get historical features
    order_items_agg = order_items.groupby(['user_id', 'product_id']).agg({
        'add_to_cart_order': ['mean', 'std'],
        'reordered': ['sum', 'mean'], 
        'order_id': 'count'
    }).reset_index()
    order_items_agg.columns = [
        'user_id', 'product_id', 
        'mean_add_to_cart_order', 'std_add_to_cart_order',
        'sum_reordered', 'mean_reordered', 
        'num_orders_with_product'
    ]
    
    # Merge aggregated order items with main data

In [9]:
py_code_saver.save("merge", merge_code)

In [ ]:
# 1. Компактный профилировщик (экономит токены, сохраняет сигнал)
def build_compact_profile(df: pd.DataFrame, target_col: str = "target") -> str:
    lines = [f"📊 SHAPE: {df.shape[0]} строк, {df.shape[1]} колонок"]
    
    if target_col in df.columns:
        dist = df[target_col].value_counts(normalize=True).to_dict()
        lines.append(f"🎯 TARGET: {dist}")
        
    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    
    # Убираем явные ID и сам таргет из списка
    skip = {"row_id", "id", target_col, "index"}
    num_cols = [c for c in num_cols if c not in skip]
    cat_cols = [c for c in cat_cols if c not in skip]
    
    lines.append(f"\n🔢 NUMERIC ({len(num_cols)}):")
    for c in num_cols[:15]:  # лимит для экономии токенов
        s = df[c].describe()
        lines.append(f"  {c}: μ={s['mean']:.2f} σ={s['std']:.2f} min={s['min']:.2f} max={s['max']:.2f} NaN={df[c].isna().mean():.1%}")
        
    lines.append(f"\n📝 CATEGORICAL ({len(cat_cols)}):")
    for c in cat_cols[:10]:
        top = df[c].value_counts().head(2).to_dict()
        lines.append(f"  {c}: uniq={df[c].nunique()} top={top} NaN={df[c].isna().mean():.1%}")
        
    return "\n".join(lines)

def extract_code(text: str) -> str:
    match = re.search(r"```(?:python)?\n(.*?)\n```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

# 3. Цикл генерации → исполнение → фидбек
def run_feature_phase(task_desc: str, df: pd.DataFrame, llm: GigaChat, target_col: str = "target"):
    profile = build_compact_profile(df, target_col)
    prompt = FEATURE_PROMPT.format(task_desc=task_desc, profile=profile, target_col=target_col)
    
    for attempt in range(1, 4):
        print(f"\nГенерация фич: попытка {attempt}")
        resp = llm.invoke([HumanMessage(content=prompt)])
        code = extract_code(resp.content if hasattr(resp, 'content') else str(resp))
        
        try:
            ns = {"pd": pd, "np": np}
            exec(code, ns)
            func = ns["generate_features"]
            
            df_res, new_cols = func(df.copy())
            # new_cols = [c for c in df_res.columns if c not in df.columns]
            
            # if len(new_cols) == 0:
            #     raise ValueError("Функция не создала новых колонок")
            # if len(new_cols) != 5:
            #     raise ValueError(f"Создано {len(new_cols)} фич, нужно ровно 5: {new_cols}")
                
            print(f"Успех! Новые колонки: {new_cols}")
            return df_res, code, new_cols
            
        except Exception as e:
            print(f"Ошибка: {e}")
            line_no, line_text = extract_exec_error(code, e)
            prompt += f"\n\n[ОШИБКА]: {e} в строке {line_no}: {line_text}\nИсправь код и верни заново."
            
    raise RuntimeError("Не удалось сгенерировать 5 фич за 3 попытки")

In [11]:
df_with_features, feature_code, new_cols = run_feature_phase(
    task_desc=description,
    df=m_train,
    llm=llm,
    target_col="target"
)

C:\Users\Yanovich\AppData\Local\Temp\ipykernel_18216\2607192458.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()



🔄 Генерация фич: попытка 1
✅ Успех! Новые колонки: ['is_frequent_buyer', 'has_large_basket', 'is_new_user', 'log_num_orders_with_product', 'log_total_distinct_products', 'order_freq_per_product', 'basket_density', 'product_popularity', 'is_morning_shopper', 'is_weekend_buyer', 'is_cold_start']


<string>:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
<string>:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting va

In [12]:
py_code_saver.save("feature_engineering", feature_code)

In [13]:
df_with_features

,row_id,user_id,product_id,target,mean_add_to_cart_order,std_add_to_cart_order,sum_reordered,mean_reordered,num_orders_with_product,total_orders,...,has_large_basket,is_new_user,log_num_orders_with_product,log_total_distinct_products,order_freq_per_product,basket_density,product_popularity,is_morning_shopper,is_weekend_buyer,is_cold_start
0,1,1,196,1,1.400000,0.966092,9,0.900000,10,10,...,0,0,2.397895,2.944439,1.000000,12.508475,0.900000,1,0,0
1,2,1,10258,1,3.333333,1.322876,8,0.888889,9,10,...,0,0,2.302585,2.944439,0.900000,12.508475,0.800000,1,0,0
2,3,1,12427,0,3.300000,2.406011,9,0.900000,10,10,...,0,0,2.397895,2.944439,1.000000,12.508475,0.900000,1,0,0
3,4,1,13032,1,6.333333,1.527525,2,0.666667,3,10,...,0,0,1.386294,2.944439,0.300000,12.508475,0.200000,1,0,0
4,5,1,13176,0,6.000000,2.828427,1,0.500000,2,10,...,0,0,1.098612,2.944439,0.200000,12.508475,0.100000,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571191,571192,206205,22035,1,7.500000,2.121320,1,0.500000,2,3,...,0,1,1.098612,3.218876,0.666667,6.000000,0.333333,0,0,0
571192,571193,206205,27845,1,1.333333,0.577350,2,0.666667,3,3,...,0,1,1.386294,3.218876,1.000000,6.000000,0.666667,0,0,0
571193,571194,206205,38739,0,7.000000,0.000000,1,0.500000,2,3,...,0,1,1.098612,3.218876,0.666667,6.000000,0.333333,0,0,0
571194,571195,206205,39160,0,9.000000,5.656854,1,0.500000,2,3,...,0,1,1.098612,3.218876,0.666667,6.000000,0.333333,0,0,0


In [ ]:
# phase3_fast_cv.py
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score


def select_top5_features_fast(
    df: pd.DataFrame,
    target_col: str = "target",
    max_sample: int = 100000,
    time_budget: float = 25.0,
) -> tuple[list[str], pd.Series, float]:
    """
    Быстрый отбор 5 лучших признаков через CatBoost CV.
    Возвращает: (top_5_names, feature_importance_series, cv_auc)
    """
    start = time.time()

    # 1. Стратифицированный сэмпл (сохраняет распределение target)
    n = min(len(df), max_sample)
    idx = np.random.RandomState(42).choice(len(df), n, replace=False)
    df_s = df.iloc[idx].copy()

    # Убираем метаданные, которые скоринг-движок всё равно отфильтрует
    drop_cols = {target_col, "row_id", "user_id", "product_id", "id", "index"}
    X = df_s.drop(columns=[c for c in drop_cols if c in df_s.columns], errors="ignore")
    y = df_s[target_col]

    # 2. Предобработка под стиль официального scoring.py
    for col in X.columns:
        if X[col].dtype == "object":
            X[col] = X[col].fillna("__UNKNOWN__").astype(str)
        else:
            X[col] = (
                pd.to_numeric(X[col], errors="coerce")
                .fillna(-999)
                .replace([np.inf, -np.inf], -999)
            )

    cat_indices = [i for i, c in enumerate(X.columns) if X[c].dtype == "object"]

    # 3. Параметры "быстрого ранжировщика"
    # Глубина 4 + 150 итераций = быстрый захват основных паттернов без переобучения на шум
    params = {
        "iterations": 150,
        "depth": 4,
        "learning_rate": 0.1,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "silent": True,
        "random_seed": 42,
        "auto_class_weights": "Balanced",  # Точно как в scoring.py
        "thread_count": -1,
    }

    CATBOOST_PARAMS = {
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3,
        "random_seed": 42,
        "verbose": 0,
        "thread_count": 1,
        "eval_metric": "AUC",
        "auto_class_weights": "Balanced",
    }

    params = CATBOOST_PARAMS

    # 4. Быстрая 3-Fold CV (5-fold слишком долго, 3-fold даёт стабильный сигнал за ~0.8с/фолд)
    cv_auc = cross_val_score(
        CatBoostClassifier(**params), X, y, cv=3, scoring="roc_auc", n_jobs=1
    ).mean()

    # 5. Фит на сэмпле для расчёта importance
    model = CatBoostClassifier(**params)
    model.fit(X, y, cat_features=cat_indices if cat_indices else None, verbose=False)

    fi = pd.Series(model.get_feature_importance(), index=X.columns).sort_values(
        ascending=False
    )
    top5 = fi.index[:5].tolist()

    elapsed = time.time() - start

    print(f"\nFast CV completed in {elapsed:.1f}s")
    print(f"   CV AUC: {cv_auc:.4f}")
    print(f"   Top 5: {top5}")

    return top5, fi, cv_auc

In [15]:
output_df = df_with_features[[*new_cols, "target"]]

In [16]:
# После генерации 10 фич (phase2)

# Быстрый отбор
top5, fi, cv_auc = select_top5_features_fast(output_df, target_col="target")

# # Сохраняем только топ-5
# final_train = merged_train.copy()
# final_test = merged_test.copy()
# for col in top5:
#     final_train[col] = df_with_10[col]
    # Для test перегенерируем фичи тем же кодом, но оставляем только топ-5
    # (или просто копируем, если фичи row-wise и не зависят от train-статистик)


⚡ Fast CV completed in 30.3s
   📊 CV AUC: 0.7564
   🏆 Top 5: ['product_popularity', 'order_freq_per_product', 'log_total_distinct_products', 'basket_density', 'log_num_orders_with_product']


In [28]:
# phase4_advisor.py
def generate_advice(task_desc: str, df_profile: str, cv_auc: float, 
                    feature_importance: dict, top5: list[str], llm: GigaChat) -> str:
    """Генерирует текстовые рекомендации на основе CV CatBoost."""
    fi_sorted = sorted(feature_importance.items(), key=lambda x: -x[1])
    top_str = ", ".join([f"{k} ({v:.1f}%)" for k,v in fi_sorted[:3]])
    weak_str = ", ".join([f"{k} ({v:.1f}%)" for k,v in fi_sorted[-3:]])

    prompt = f"""Ты — Senior ML Analyst. Проанализируй результаты генерации признаков для CatBoost.
ЗАДАЧА: {task_desc}
ПРОФИЛЬ ДАННЫХ: {df_profile}
БЫЛИ СОЗДАНЫ НОВЫЕ ФИЧИ: {feature_code}
РЕЗУЛЬТАТЫ: CV AUC = {cv_auc:.4f} | Топ: {top_str} | Слабые: {weak_str}

ПРАВИЛА:
1. Объясни, какой бизнес-сигнал ловят топ-фичи.
2. Укажи 2-3 пропущенных паттерна (статусы, магические числа, отношения счётчиков, пороги).
3. Дай РОВНО 3 конкретных совета для создания новых фич. Формулируй как команды: "создай флаг...", "возьми log1p...", "раздели..." итд.
4. Советы ОБЯЗАНЫ быть совместимы с деревьями: БЕЗ нормализации, БЕЗ factorize(), БЕЗ деления на mean/std.
5. Верни ТОЛЬКО текст рекомендаций. Без markdown, без вступлений, без списков. Максимум 200 слов."""

    resp = llm.invoke([SystemMessage(content=prompt), HumanMessage(content="Проанализируй и верни рекомендации.")])
    return resp.content if hasattr(resp, 'content') else str(resp)

In [ ]:
profile = build_compact_profile(m_train, "target")
advice = generate_advice(task_desc=description, df_profile=profile, 
                         cv_auc=cv_auc, feature_importance=dict(fi), top5=top5, llm=llm)
print(f"Рекомендации:\n{advice}\n")

C:\Users\Yanovich\AppData\Local\Temp\ipykernel_18216\2607192458.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


💡 Рекомендации:
Топовые признаки хорошо отражают поведение покупателей:
- product_popularity показывает относительную популярность продукта среди всех заказов;
- order_freq_per_product отражает частоту покупки конкретного товара относительно общего количества заказов пользователя;
- log_total_distinct_products учитывает разнообразие покупок в логарифмической шкале, сглаживая влияние редких случаев большого ассортимента.

Пропущенные паттерны:
- Статус первого заказа или первой покупки данного товара;
- Магическое число среднего времени между покупками одного товара;
- Отношение общей суммы товаров к количеству уникальных продуктов.

Рекомендации по новым признакам:
Создай флаг first_purchase, равный единице если это первый заказ товара пользователем.
Возьми log1p от средней стоимости корзины, чтобы учесть её размер в логарифмическом масштабе.
Раздели количество повторных заказов на общее количество дней активности пользователя, получая среднюю частоту повторного заказа за день.



In [ ]:
# Базовый промпт (ваша текущая версия)
BASE_FEATURE_PROMPT = """Ты — Senior ML Engineer. Пишешь ТОЛЬКО python код.
ЗАДАЧА: Создать РОВНО 10 новых признаков для задачи: {task_desc}
ПРОФИЛЬ ДАННЫХ: {profile}

⛔ ЖЁСТКИЕ ОГРАНИЧЕНИЯ:
1. Начни с: df = df.copy()
2. Заполни пропуски: числа → 0, строки → 'UNKNOWN'.
3. ЗАПРЕЩЕНО: groupby, merge, transform, rolling, shift, apply.
4. ЗАПРЕЩЕНО использовать: target, row_id, id, uuid в вычислениях.
5. Избегай np.inf: при делении добавляй +1e-9.
6. Верни ТОЛЬКО код функции. Без markdown, без комментариев.

🌲 ПРАВИЛА ДЛЯ CATBOOST:
1. НЕ нормализуй, НЕ дели на mean/max/std. Деревья инвариантны к масштабу.
2. НЕ используй .factorize() или арифметику над категориями.
3. Для категорий/статусов → бинарные флаги (col == 'x').astype(int).
4. Для магических чисел (999, -1) → ТОЛЬКО флаги наличия.
5. Приоритет: [Статусные флаги] > [log1p счётчиков] > [Взаимодействия числовых] > [Бинарные пороги].

ФОРМАТ:
def generate_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    ... код ...
    return df, new_columns"""

def extract_code(text: str) -> str:
    m = re.search(r"```(?:python)?\n(.*?)\n```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def run_phase2_with_advice(task_desc: str, df: pd.DataFrame, llm: GigaChat, advice: str = ""):
    profile = "\n".join([f"{c}: dtype={df[c].dtype}, nunique={df[c].nunique()}, NaN%={df[c].isna().mean():.1%}" 
                         for c in df.select_dtypes(include=['number','object']).columns[:15]])
    
    prompt = BASE_FEATURE_PROMPT.format(task_desc=task_desc, profile=profile)
    
    # 🔑 Инъекция советов
    if advice.strip():
        prompt += f"\n\n⚡ [УЧТИ РЕКОМЕНДАЦИИ АНАЛИТИКА]:\n{advice}\nПримени их, строго соблюдая ⛔ ограничения."

    messages = [SystemMessage(content=prompt), HumanMessage(content="Начни. Верни ТОЛЬКО код.")]
    
    for attempt in range(1, 6):
        print(f"\nГенерация (итерация 2): попытка {attempt}/5")
        try:
            resp = llm.invoke(messages)
            code = extract_code(resp.content if hasattr(resp, 'content') else str(resp))
            
            ns = {"pd": pd, "np": np}
            exec(code, ns)
            df_res, new_cols = ns["generate_features"](df.copy())
            # new_cols = [c for c in df_res.columns if c not in df.columns]
            
            assert len(new_cols) >= 8, f"Слишком мало фич: {len(new_cols)}"
            print(f"Успех! Сгенерировано {len(new_cols)} фич.")
            return code, df_res, new_cols
            
        except Exception as e:
            # Ваша extract_exec_error логика здесь
            err_msg = f"{type(e).__name__}: {e}"
            print(f"{err_msg}")
            if attempt == 5: raise RuntimeError("Генерация провалилась")
            messages.append(HumanMessage(content=f"[ОШИБКА]: {err_msg}\nИсправь ТОЛЬКО проблемную строку. Верни полную функцию."))
    return None, None, []

In [ ]:
code2, df2, cols2 = run_phase2_with_advice(task_desc=description, df=m_train, llm=llm, advice=advice)

# 4. Финальный отбор топ-5 из улучшенного пула
top5_final, fi_final, cv_final = select_top5_features_fast(df2, max_sample=1000000)
print(f"\nФинальные топ-5: {top5_final} | CV AUC: {cv_final:.4f}")

C:\Users\Yanovich\AppData\Local\Temp\ipykernel_18216\29708357.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=['number','object']).columns[:15]])



🔄 Генерация (итерация 2): попытка 1/5
✅ Успех! Сгенерировано 10 фич.

⚡ Fast CV completed in 219.9s
   📊 CV AUC: 0.7608
   🏆 Top 5: ['product_popularity', 'order_freq_per_product', 'department_id', 'log_total_sum_over_unique', 'reordered_share']

🏆 Финальные топ-5: ['product_popularity', 'order_freq_per_product', 'department_id', 'log_total_sum_over_unique', 'reordered_share'] | CV AUC: 0.7608


In [39]:
top5_final

['product_popularity',
 'order_freq_per_product',
 'department_id',
 'log_total_sum_over_unique',
 'reordered_share']

In [73]:
from src.main.utils.final_exporter import export_final_output

In [79]:
# 1. Применяем сгенерированный код к данным после Merge
ns = {"pd": pd, "np": np}
exec(code2, ns)
train_full, _ = ns["generate_features"](m_train.copy())
test_full, _  = ns["generate_features"](m_test.copy())

# 2. Собираем колонки: ВСЕ оригинальные + ТОП-5 новых (дубли исключаются автоматически)
final_train_cols = list(train.columns) + [c for c in top5_final if c not in train.columns]
final_test_cols  = list(test.columns)  + [c for c in top5_final if c not in test.columns]

In [80]:
train_full.columns

Index(['row_id', 'user_id', 'product_id', 'target', 'mean_add_to_cart_order',
       'std_add_to_cart_order', 'sum_reordered', 'mean_reordered',
       'num_orders_with_product', 'total_orders', 'avg_days_between_orders',
       'avg_basket_size', 'total_distinct_products', 'reordered_share',
       'favorite_day_of_week', 'favorite_hour_of_day', 'product_name',
       'aisle_id', 'department_id', 'aisle', 'department', 'first_purchase',
       'log_avg_basket_size', 'daily_reorder_rate', 'high_activity_user',
       'log_total_sum_over_unique', 'avg_time_between_purchases',
       'product_popularity', 'order_freq_per_product', 'days_of_week_variety',
       'log_total_distinct_products'],
      dtype='str')

In [81]:
final_train_cols

['row_id',
 'user_id',
 'product_id',
 'target',
 'product_popularity',
 'order_freq_per_product',
 'department_id',
 'log_total_sum_over_unique',
 'reordered_share']

In [82]:
final_test_cols

['row_id',
 'user_id',
 'product_id',
 'target',
 'product_popularity',
 'order_freq_per_product',
 'department_id',
 'log_total_sum_over_unique',
 'reordered_share']

In [83]:
# 3. Формируем финальные датафреймы
final_train = train_full[final_train_cols]
final_test  = test_full[final_test_cols]

In [ ]:

# 4. Категорики → str (чтобы scoring.py сам подхватил их в cat_features)
for col in top5:
    if col in final_train.columns and final_train[col].dtype == "object":
        final_train[col] = final_train[col].astype(str)
        final_test[col]  = final_test[col].astype(str)

print(f"Готово. Train: {final_train.shape} | Test: {final_test.shape}")
print(f"Итоговые колонки: {list(final_train.columns)}")

# 5. Сохранение
os.makedirs("output", exist_ok=True)
final_train.to_csv("output/train.csv", index=False)
final_test.to_csv("output/test.csv", index=False)

✅ Готово. Train: (571196, 9) | Test: (144846, 9)
📋 Итоговые колонки: ['row_id', 'user_id', 'product_id', 'target', 'product_popularity', 'order_freq_per_product', 'department_id', 'log_total_sum_over_unique', 'reordered_share']
